In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Load Titanic
df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

features = ["Pclass", "Sex", "Age", "Fare", "FamilySize", "IsAlone"]
X = df[features]
y = df["Survived"]

# Define parameters to try
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10]
}

# GridSearchCV
model = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(model, param_grid, 
                           cv=5, scoring='accuracy')
grid_search.fit(X, y)

# Best parameters
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

Best Parameters: {'max_depth': 7, 'min_samples_split': 2, 'n_estimators': 50}
Best Score: 0.8361810306948717


In [5]:
from sklearn.model_selection import train_test_split

# Use best parameters
best_model = RandomForestClassifier(
    max_depth=7,
    min_samples_split=2,
    n_estimators=50,
    random_state=42
)

# Load test data
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
test["Age"] = test["Age"].fillna(test["Age"].median())
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)
test["Fare"] = test["Fare"].fillna(test["Fare"].median())

# Train on full data
best_model.fit(X, y)

# Submit
predictions = best_model.predict(test[features])
output = pd.DataFrame({
    'PassengerId': test.PassengerId,
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)
print("Done!")

Done!
